# Building GPT

## 1. 准备数据

下载数据并查看

In [6]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://ghproxy.net/https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-04 19:57:04--  https://ghproxy.net/https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving ghproxy.net (ghproxy.net)... 51.195.241.253
Connecting to ghproxy.net (ghproxy.net)|51.195.241.253|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M   246KB/s    in 4.4s    

2026-03-04 19:57:16 (246 KB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [1]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [3]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



数据处理，采用最简单的编解码方式，逐个字符对应，共65个字符；



In [4]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))  # set去重/形成列表/排序
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


也有其他的token处理方式，基于句子或者基于单词，如GPT实际使用的fast BPE tokenizer，例如：

In [5]:
import tiktoken
tik_encoder = tiktoken.get_encoding("gpt2")
print(tik_encoder.encode("hello world"))
print(tik_encoder.decode(tik_encoder.encode("hello world")))

[31373, 995]
hello world


In [6]:
# 创建字符和数字的映射关系，由此构建编码和解码器
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]  # 看往后预测一个元素

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

划分为block，范围内的作为同一批次训练样本，基于自回归可以通过已知tokens预测下一token，再将其作为已知tokens继续预测，如：

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


### 数据加载


In [11]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [12]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


## 模型

1. 需要reshape，因为cross_entropy要求输入是二维的，第一维是样本数量，第二维是类别数量，而现在的logits是三维的，第一维是批量大小，第二维是时间步，第三维是类别数量，所以我们需要把前两维合并成一维，这样每个时间步的预测都被视为一个独立的样本。
2. 监督学习需要有对应的targets来进行损失的计算

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # 转化为一个(vocab_size, vocab_size)的矩阵，行是输入字符，列是输出字符，值是对应的logit
        # 我们要预测下一个字符的概率分布，而这个分布的维度就是vocab_size，所以每个输入字符都对应一个长度为vocab_size的向量，表示对每个可能输出字符的logit值。
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)  # 
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # 接受idx和max_new_tokens参数，idx是当前上下文的索引数组，max_new_tokens是要生成的新令牌的数量
    def generate(self, idx, max_new_tokens):
        # idx是一个(B, T)的张量，表示当前上下文的索引数组，B是批量大小，T是当前上下文的长度，也可以认为是时间步
        for _ in range(max_new_tokens):
            # 调用前向方法获取当前上下文的logits和损失
            logits, loss = self(idx)
            # 只关注最后一个时间步的logits，因为只预测下一个token
            logits = logits[:, -1, :] # becomes (B, C)
            # 转换为概率
            probs = F.softmax(logits, dim=-1) # (B, C)
            # 从概率分布中采样一个索引作为下一个token
            # 为什么采样？因为我们希望生成的文本具有多样性，而不是每次都选择概率最高的令牌，这样可以避免生成重复和无趣的文本。
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # 将新生成的索引追加到当前上下文的idx中，形成新的上下文，以便在下一次迭代中使用
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# 0作为起始token，生成100个新token；索引0取batch的第0个样本；tolist将生成的索引列表转换为Python列表；decode将索引列表转换为字符串
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [15]:
batch_size = 32
for steps in range(1000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    # 上一步梯度清零，set_to_none=True可以稍微加速训练；获取参数梯度；更新参数
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

3.704136610031128


In [16]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Wh;;Sq.f ustNzknc
kwgOj$dhPWr,SV?hsusiKpgXXUh;Apmem d?hESXI.i;TrJgkiF-oKbXCAA -botrngFCHAUQkn$

pn$w-gHoi?wtd!
LLULIfSK'bAw :M.ZtOptXEQcL?hfaofqbPd?OnonQQJMap$aypupIBYGUsZaI'ottllo..k$W$Akp?yl?ajKlzY!lx&QQLW? t,bXFkyhl-dmVsHeckhRl,jSClgjuk:3Iv
?OqlrV;!Plxfzgy;;
'mRjuBQ&xk!$
h
SiruDJgKuDny,S$ERf.?GSV-ivvKcOvi-nQGX&q-YQbm dEM?px;Akr-IESq--wIWId
RFgXTpDUgM:CK$I!uo'IBT -
j?wfy fFr.&fiqtRS.ZttxGh' a!ogrn$zoZqbocL&yIffBDWNUboscuQqo.Fls,?,M?eZxHx?p?EV.mJiHqHnxT  bQpa;P fawiF$-QbWv&f:CVDCBfano,b?$Esev.?


## 3. 自注意力

自注意力的数学trick
1）预测下一个token需要与前面的tokens进行交互，可以采取求和的方式

In [17]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))  # 生成下三角矩阵
a = a / torch.sum(a, 1, keepdim=True)  # 每行除以该行的和，使得每行的元素之和为1
b = torch.randint(0,10,(3,2)).float()
c = a @ b  # 矩阵乘法
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


上面需要加keepdim=True，下面举例说明：

In [18]:
a = torch.tensor([[1., 0., 0.],
                  [1., 1., 0.],
                  [1., 1., 1.]])

# 不加
sum_without = torch.sum(a, 1)        # dim=1 表示按行求和
print(sum_without.shape)             # torch.Size([3])  ← 一维！
print(sum_without)                   # tensor([1., 2., 3.])
# 尝试做除法
a / sum_without
# 结果：广播失败或结果错误 ❌
# PyTorch 会把 [3] 广播成 [3,3]，变成列方向除，不是我们要的行归一化

torch.Size([3])
tensor([1., 2., 3.])


tensor([[1.0000, 0.0000, 0.0000],
        [1.0000, 0.5000, 0.0000],
        [1.0000, 0.5000, 0.3333]])

In [19]:
sum_with = torch.sum(a, 1, keepdim=True)
print(sum_with.shape)                # torch.Size([3, 1])  ← 二维！
print(sum_with)                      # tensor([[1.], [2.], [3.]])

# 做除法
a / sum_with
# 结果：正确！✅
# (3,3) / (3,1) → 每行分别除以自己的行和

torch.Size([3, 1])
tensor([[1.],
        [2.],
        [3.]])


tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [20]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 32])

In [21]:
xbow = torch.zeros((B, T, C))  # 初始化为0
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]  # (t, C) 当前时间步之前的所有输入
        xbow[b, t] = torch.mean(xprev, dim=0)  # 计算当前时间步之前的所有输入的平均值

通过下三角矩阵的方式能够模拟下一输入只取决于前面的tokens

In [22]:
wei = torch.tril(torch.ones(T, T))  # 生成下三角矩阵
wei = wei / wei.sum(1, keepdim=True)  # 每行除以该行的和，使得每行的元素之和为1
xbow2 = wei @ x  # (B,T,T) @ (B,T,C) ---> (B,T,C)
torch.allclose(xbow, xbow2)  # 验证xbow和xbow2是否近似相等

True

masked_fill使所有等于0的元素都为负无穷，然后再经过softmax就相当于行归一化了，可以下面一行一行运行输出wei看wei的变化：

In [23]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

### 单头自注意力实现
1、k和q相乘可以得到原始的亲和力；掩码后该位置之后的tokens不允许被看到；softmax之后可以看到每一个token聚合了多少信息

2、这里实现的是一个decoder，因为当前token无法看到未来的tokens；如果想要实现encoder，也就是希望所有tokens之间能够互通，看得到未来的tokens，则删除wei = wei.masked_fill(tril == 0, float('-inf'))，比如情感分析的场景，我们需要分析整句话，而不是句子生成

3、当前属于self-attention，因为key、query、value来自于同一个值x，希望相互查看和交流；attention更常用，其key来自于x，而query、value来自于一个独立的外部源，比如我们关心的其他上下文；cross-attention则是希望从另一个独立的节点提取信息到我们的节点

4、q与k的转置相乘之后，必须要进行归一化，否则如果值之间相差很大的话，进行softmax会向数值高的方向集中，导致接近one-hot向量，使得只能从单个节点聚合信息，这不得行

In [33]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
v = value(x)
wei = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, 16) @ (B, 16, T) ---> (B, T, T)  这里表示-2和-1列位置交换

In [32]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T))  # 初始为0（这只是举个例子，真实的是上一块的最后一行），后面会变，根据大小不同注意力也不同（亲和力？）
wei = wei.masked_fill(tril == 0, float('-inf'))  # 没有未来token的信息，填充-inf
wei = F.softmax(wei, dim=-1)  # 行归一化
# out = wei @ x  # 对过去的token和当前token进行了一个简单平均（举个例子，下一行才是真正应该执行的操作）
out = wei @ v

wei

tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [6.5128e-01, 3.4872e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [3.6389e-01, 3.6104e-01, 2.7507e-01, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [6.8366e-01, 1.1415e-01, 1.4943e-01, 5.2752e-02, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [6.3201e-02, 3.6178e-01, 1.0332e-01, 3.0479e-01, 1.6690e-01,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [4.1383e-01, 1.1322e-01, 9.7611e-02, 5.5666e-02, 5.5912e-02,
          2.6376e-01, 0.0000e+00, 0.0000e+00],
         [4.1486e-01, 1.9180e-01, 8.4875e-02, 1.2112e-01, 8.6212e-02,
          5.3236e-02, 4.7899e-02, 0.0000e+00],
         [1.2598e-01, 5.2171e-02, 1.8922e-01, 3.6297e-01, 9.4476e-02,
          4.7452e-02, 6.0716e-02, 6.7011e-02]],

        [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.00

神经网络深度过深的话会遇到优化问题，transformers采用了两个措施来解决：
1、残差连接
2、层归一化

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

以上的那些实现都是仅解码器的，而transformers用于机器翻译，是encoder-decoder结构，有如下所示的特殊字符进行标记：<start>

In [ ]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>